# C3 / C4 — GPU-добор для статьи

Обучает две оставшиеся глубокие конфигурации — `C3_pure_cnn` и
`C4_pure_transformer` — на GPU. Тот же код и те же данные, что в основном
прогоне: отличается только устройство, а оно на метрику не влияет.

Встроены три стража сопоставимости — 67 признаков в том же порядке, размеры
срезов 89 973 / 11 247 и словарь ровно 498 токенов. Если хоть что-то
разойдётся с прогоном на Kaggle, ноутбук остановится с внятным сообщением,
а не выдаст молча числа, которые нельзя положить в одну таблицу с
остальными восемью конфигурациями.

## Что нужно сделать

1. **Runtime → Change runtime type → GPU** (подойдёт T4), затем **Connect**.
2. **Runtime → Run all**.
3. На третьей ячейке появится кнопка выбора файла — загрузи свой
   `kaggle.json` (Kaggle → аватар → Settings → API → *Create New Token*).
   Это твой токен: он нужен, чтобы скачать два `parquet` из твоего
   приватного датасета, и остаётся только внутри этой сессии.
4. В конце ноутбук напечатает блок между `RESULTS_JSON_START` и
   `RESULTS_JSON_END` — **скопируй его целиком обратно в чат Claude**.

Время: примерно час-полтора на конфигурацию, итого 2–3 часа. Каждая
сохраняется сразу по готовности, поэтому обрыв сессии не уничтожает уже
досчитанное — при повторном запуске готовые конфигурации пропускаются.

In [ ]:
# 1. Проверка, что GPU действительно выделен
import torch
assert torch.cuda.is_available(), (
    "GPU не выделен! В Lightning выбери GPU-машину справа (L4); "
    "в Colab: Runtime -> Change runtime type -> GPU. Затем запусти заново.")
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

In [ ]:
# 2. Публичный код проекта + зависимости
!git clone --depth 1 -b v2-restructure https://github.com/SergeySolovyev/smart-contract-vuln-detection-from-bytecode.git repo
# kaggle пинуем на 1.8.2 -- именно эта версия работает с этим датасетом на
# машине автора. Версия из Colab по умолчанию давала 403 на приватном датасете.
!pip -q install "xgboost" "scikit-learn" pyarrow joblib "kaggle==1.8.2"
import os
print("dl_pipeline на месте:", os.path.exists("repo/src/dl_pipeline.py"))

In [ ]:
# 3. Данные: два parquet из твоего приватного Kaggle-датасета.
import os, glob, json, zipfile, pathlib, shutil

home_kaggle = pathlib.Path.home() / ".kaggle" / "kaggle.json"
try:
    from google.colab import files
    print("Загрузи kaggle.json (Kaggle -> аватар -> Settings -> API -> Create New Token):")
    up = files.upload()
    home_kaggle.parent.mkdir(parents=True, exist_ok=True)
    home_kaggle.write_bytes(list(up.values())[0])
except ImportError:
    assert home_kaggle.exists(), "положи kaggle.json в ~/.kaggle/"
os.chmod(home_kaggle, 0o600)

# Переменные окружения ПЕРЕБИВАЮТ файл в клиенте Kaggle. Если Colab выставил
# свои (пустые или чужие), файл, который мы только что записали, был бы молча
# проигнорирован -- поэтому задаём явно и убираем перенаправление конфига.
cred = json.loads(home_kaggle.read_text())
os.environ["KAGGLE_USERNAME"] = cred["username"]
os.environ["KAGGLE_KEY"] = cred["key"]
os.environ.pop("KAGGLE_CONFIG_DIR", None)
print("аутентификация под:", cred["username"])

DS = "sergeisolovyev/defi-bytecode-features-v2"
WANT = ["train_v2.parquet", "test_v2.parquet"]
os.makedirs("data_v2", exist_ok=True)

# Через Python API, а не через ! -- так ошибка поднимается исключением,
# а не теряется в выводе, оставляя пустую папку и невнятный сбой ниже.
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
try:
    listing = [f.name for f in api.dataset_list_files(DS).files]
    print("датасет виден, файлов:", len(listing))
except Exception as e:
    raise SystemExit(
        f"Kaggle отказал: {e}\n"
        "Токен и датасет проверены и рабочие с машины автора, значит дело в "
        "среде Colab. Запасной путь: сделать датасет публичным (одна "
        "переключалка на его странице) либо положить два parquet на Google "
        "Drive и смонтировать его. Сообщи об этом Claude.")

for f in WANT:
    if os.path.exists(f"data_v2/{f}"):
        print(f"{f}: уже скачан")
        continue
    print(f"качаю {f} ...")
    api.dataset_download_file(DS, f, path="data_v2", force=True)

for z in glob.glob("data_v2/*.zip"):
    zipfile.ZipFile(z).extractall("data_v2")
    os.remove(z)

for f in WANT:
    p = f"data_v2/{f}"
    assert os.path.exists(p), f"{f} не скачался"
    print(f"  {f}: {os.path.getsize(p)/1e6:.0f} MB")
print("данные готовы")

In [ ]:
# 4. Обучение C3 и C4 на GPU (тот же вызов, что в основном пайплайне)
import sys, time, json, pathlib
import numpy as np, pandas as pd, torch
sys.path.insert(0, "repo/src")
from dl_pipeline import (BytecodeTokenizer, build_token_ids,
                         DL_EXPERIMENTS, run_dl_experiment)

LABELS = ["access-control", "arithmetic", "bad-randomness", "double-spending",
          "locked-ether", "other", "reentrancy", "unchecked-calls"]
FEATURES = json.load(open("repo/data/feature_columns.json"))
WANT = {"C3_pure_cnn", "C4_pure_transformer"}

# Guards: these two configs must be comparable with the 8 already trained on
# Kaggle, so the inputs have to be IDENTICAL, not merely similar. Verified
# against the Kaggle trainer: same 67 features in the same order, tokenizer
# fit on exactly the first 20000 train rows -> vocabulary of 498 tokens.
# If either differs, stop loudly rather than emit numbers that silently
# cannot be compared with the rest of the ablation.
assert len(FEATURES) == 67, f"expected 67 features, got {len(FEATURES)}"

cols = ["bytecode"] + FEATURES + LABELS
tr = pd.read_parquet("data_v2/train_v2.parquet", columns=cols)
te = pd.read_parquet("data_v2/test_v2.parquet", columns=cols)
print(f"train {len(tr):,}  test {len(te):,}  device=cuda")
assert len(tr) == 89973 and len(te) == 11247, "unexpected split sizes"

tok = BytecodeTokenizer().fit(tr["bytecode"].head(20000))
print("vocab:", tok.vocab_size)
assert tok.vocab_size == 498, (
    f"tokenizer vocab {tok.vocab_size} != 498 expected -- inputs differ from "
    "the Kaggle run, results would NOT be comparable. Stop and report this.")
tr_ids = build_token_ids(tr["bytecode"], tok)
te_ids = build_token_ids(te["bytecode"], tok)
y_tr = tr[LABELS].to_numpy("float32"); y_te = te[LABELS].to_numpy("float32")
X_tr = tr[FEATURES].to_numpy("float32"); X_te = te[FEATURES].to_numpy("float32")
del tr, te

RUNS = pathlib.Path("runs_out"); RUNS.mkdir(exist_ok=True)
out = {}
for cfg in DL_EXPERIMENTS:
    if cfg["name"] not in WANT:
        continue
    print(f"\n=== {cfg['name']} ===")
    t0 = time.time()
    res = run_dl_experiment(cfg, tok, tr_ids, y_tr, X_tr, te_ids, y_te, X_te,
                            runs_dir=RUNS, wandb_run=None)
    mf = res.get("macro_f1_external", res.get("macro_f1"))
    print(f"  {cfg['name']} macro_f1={mf:.4f}  ({(time.time()-t0)/60:.0f} min)")
    out[cfg["name"]] = res
print("\nобе конфигурации обучены")

In [ ]:
# 5. Отдать результат: JSON для копипаста в чат + сохранённые файлы
import json
for name, res in out.items():
    open(f"{name}.json", "w").write(json.dumps(res))
print("\n=========== СКОПИРУЙ ВСЁ МЕЖДУ МЕТКАМИ ОБРАТНО В ЧАТ CLAUDE ===========\n")
print("RESULTS_JSON_START")
print(json.dumps(out))
print("RESULTS_JSON_END")
print("\nТакже сохранены файлы:", [f"{n}.json" for n in out],
      "\n(их можно скачать из файлового браузера Studio, но хватит и JSON выше)")